<a href="https://colab.research.google.com/github/Preet-Kanwal-Singh/Machine-Learning-Projects/blob/main/MentalHealth/MentalHealthAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unidecode
!pip install contractions
!pip install nltk

In [ ]:
import pandas as pd

import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

import re
import string

from unidecode import unidecode
import contractions

df = pd.read_csv('/content/Combined Data.csv')
print(df.shape)
print(df.columns)
print(df.isnull().sum())
df.head()

In [ ]:
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

df['statement'] = df['statement'].astype(str)  # ensure string
df['statement'] = df['statement'].replace('nan', pd.NA)  # if string 'nan' exists
df = df.dropna(subset=['statement'])
df = df[df['statement'].str.strip() != '']   # remove empty strings

print("After removing missing:", df.shape)


In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['statement', 'status'], keep='first')
after = len(df)
print(f"Removed {before-after} duplicate rows")


In [ ]:
# class distribution
counts = df['status'].value_counts()
print(counts)

min_samples = 50
rare = counts[counts < min_samples].index.tolist()
print("Rare labels:", rare)


In [ ]:
texts = df['statement'].astype(str).tolist()
labels = df['status'].astype(str).values
label_names = sorted(df['status'].unique())

tfidf = TfidfVectorizer(max_features=2000, stop_words='english')
X = tfidf.fit_transform(texts)

centroids = []
for lab in label_names:
    mask = (labels == lab)
    centroid = X[mask].mean(axis=0)
    centroids.append(np.asarray(centroid).ravel())
centroids = np.vstack(centroids)

Z = linkage(centroids, method='ward')
plt.figure(figsize=(10, 6))
dendrogram(Z, labels=label_names, leaf_rotation=30, leaf_font_size=10)
plt.title("Agglomerative Clustering Dendrogram for Status Labels")
plt.xlabel("Status Categories")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
def clean_text(text, min_len=3):
    if not isinstance(text, str):
        return ""
    # 1. normalize unicode chars
    text = unidecode(text)
    # 2. expand contractions ("don't" -> "do not")
    text = contractions.fix(text)
    # 3. lower
    text = text.lower()
    # 4. remove urls, emails, html tags
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'<.*?>', ' ', text)
    # 5. remove punctuation and numbers (keeping spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 6. collapse repeated characters (e.g., coooool -> cool)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 7. collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # 8. optionally drop very short tokens / statements
    if len(text.split()) < min_len:
        return ""
    return text

# Apply
df['cleaned'] = df['statement'].astype(str).apply(clean_text)
# drop rows turned empty after cleaning
df = df[df['cleaned'] != ""]
print("After cleaning:", df.shape)
df[['statement','cleaned','status']].head()

# Download the missing 'punkt_tab' resource
nltk.download('punkt_tab')

stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def tokenize_clean(text):
    toks = word_tokenize(text)
    toks = [t for t in toks if t.isalpha() and t not in stop]  # remove stopwords & non-alpha
    toks = [lemmatizer.lemmatize(t) for t in toks]
    return " ".join(toks)

df['clean_tokens'] = df['cleaned'].apply(tokenize_clean)
# drop empty after tokenization
df = df[df['clean_tokens'].str.strip() != ""]
print("After tokenizing & lemmatizing:", df.shape)
df[['cleaned','clean_tokens']].head()

In [ ]:
df.loc[:, 'n_tokens'] = df['clean_tokens'].str.split().apply(len)

print(df['n_tokens'].describe())

# drop too short or too long samples
df = df[(df['n_tokens'] >= 3) & (df['n_tokens'] <= 200)]

print("After length filtering:", df.shape)


In [ ]:
df.to_csv('/content/Cleaned_Combined_Data.csv', index=False)
print("Saved cleaned CSV.")

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

DATA_PATH = '/content/Cleaned_Combined_Data.csv'
df = pd.read_csv(DATA_PATH)

X = df['clean_tokens'].astype(str).values
y = df['status'].astype(str).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,          # 80% train, 20% test
    random_state=42,
    stratify=y               # preserves class distribution
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print("Vectorized shapes:", X_train_vec.shape, X_test_vec.shape)

In [ ]:
logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_train_vec, y_train)
pred_lr = logreg.predict(X_test_vec)

print("\n================ Logistic Regression ================")
print("Accuracy:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr))


In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_vec, y_train)
pred_svm = svm.predict(X_test_vec)

print("\n================ Linear SVM (LinearSVC) ================")
print("Accuracy:", accuracy_score(y_test, pred_svm))
print(classification_report(y_test, pred_svm))

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
pred_nb = nb.predict(X_test_vec)

print("\n================ Multinomial Naive Bayes ================")
print("Accuracy:", accuracy_score(y_test, pred_nb))
print(classification_report(y_test, pred_nb))

In [ ]:
# -------------------------------------------------------
# Test the model with custom examples
# -------------------------------------------------------

test_sentences = [
    "I can’t sleep and my thoughts are racing.",
    "I feel sad and hopeless lately.",
    "Everything is overwhelming, I feel too stressed.",
    "I am so happy today!",
    "My panic attacks are getting worse.",
]

# Ensure that the preprocessing for test sentences is IDENTICAL to the training data
def preprocess_for_prediction(text):
    if not isinstance(text, str):
        return ""
    # Apply clean_text steps
    text = unidecode(text)
    text = contractions.fix(text)
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Apply tokenize_clean steps
    toks = word_tokenize(text)
    # `stop` and `lemmatizer` are globally available from cell XD26fNJbhZOh
    toks = [t for t in toks if t.isalpha() and t not in stop]
    toks = [lemmatizer.lemmatize(t) for t in toks]
    return " ".join(toks)

preprocessed_samples = [preprocess_for_prediction(s) for s in test_sentences]

# Re-initialize and re-fit tfidf to ensure it matches the logreg model's expectations
# This assumes X_train and tfidf parameters are consistent with GvsbQtu1FUej
tfidf_logreg_predictor = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    stop_words='english'
)
tfidf_logreg_predictor.fit(X_train) # Fit on the original training data (X_train is available in kernel)

# Convert to TF-IDF using the correctly configured vectorizer
sample_vecs = tfidf_logreg_predictor.transform(preprocessed_samples)

# Predict
sample_preds = logreg.predict(sample_vecs)

# Show results
print("\n================ Test Run ================")
for text, pred in zip(test_sentences, sample_preds):
    print(f"Input: {text}")
    print(f"Predicted Label: {pred}")
    print("----------------------------------------")